# Class 06 · Working with zoom levels

In class 05 you used the HEALPix grid at zoom 6. The catalog has the same data at many **zoom levels**.
In this notebook you see what the zoom changes, and what it does not:

1. how many cells, how far apart and how much data each zoom has,
2. the **same map** at different zooms, for the globe and for a region (the Alps),
3. why the **global mean** is the same at every zoom,
4. why the value at **one place** is not, and
5. how to **choose** a zoom.

Data: ERA5 (2 m temperature, mean of 2021–2023) and one day of the ICON simulation ngc4008.

## 1. The zoom levels

At zoom $z$ the grid has $12 \cdot 4^z$ cells of equal area: every zoom level splits each cell into 4.
The spacing is about the square root of the area of a cell; a map of one value per cell, in 4-byte numbers,
takes 4 bytes per cell:

In [ ]:
import numpy as np

EARTH_SURFACE = 510e6  # km²

print("zoom        cells   spacing   one map")
for zoom in range(11):
    cells = 12 * 4**zoom
    spacing = np.sqrt(EARTH_SURFACE / cells)
    print(f"{zoom:4d} {cells:12,d} {spacing:7.0f} km {cells * 4 / 1e6:8.2f} MB")

Every zoom level halves the spacing and multiplies the data by 4. A year of daily maps at zoom 10 would
be 365 × 50 MB, about 18 GB.

## 2. The same map at four zooms

The catalog of class 05, and the ERA5 2 m temperature, mean of 2021–2023, at zoom 1, 3, 5 and 7. ERA5 has
monthly means, so this is little data.

In [ ]:
import intake

cat = intake.open_catalog("https://data.nextgems-h2020.eu/online.yaml")


def era5_mean(zoom):
    # ERA5 2 m temperature, mean of 2021-2023, in °C
    era5 = cat.ERA5(zoom=zoom).to_dask()["2t"]
    return era5.sel(time=slice("2021", "2023")).mean("time").compute() - 273.15

In [ ]:
import cartopy.crs as ccrs
import easygems.healpix as egh
import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    2, 2, figsize=(14, 7), subplot_kw={"projection": ccrs.Robinson()}
)
for ax, zoom in zip(axes.flat, [1, 3, 5, 7]):
    data = era5_mean(zoom)
    ax.set_global()
    ax.coastlines(linewidth=0.4)
    image = egh.healpix_show(data, ax=ax, cmap="RdYlBu_r", vmin=-30, vmax=30)
    ax.set_title(f"zoom {zoom}: {data.size:,} cells")
fig.colorbar(image, ax=axes, shrink=0.6, label="temperature / °C")

fig.savefig("/app/pages/06_zoom_levels/maps_global.png", dpi=100, bbox_inches="tight")
plt.show()

At zoom 1 you see the shape of the cells: diamonds, 4 inside each of the 12 cells of zoom 0. From zoom 5 on,
the globe looks smooth: for a map of the whole globe, zoom 5 or 6 is enough.

## 3. A region: the Alps

For a small region you need a finer zoom. One day (15 July 2021) of the ICON simulation of class 05, at
zoom 6 to 9. `set_extent` chooses the region: west, east, south, north; `BORDERS` draws the countries.

The ICON data at zoom 10 has no values of the 2 m temperature, so zoom 9 (about 12 km) is the finest here.

In [ ]:
import cartopy.feature as cfeature


def icon_day(zoom):
    # ICON ngc4008, 2 m temperature of 15 July 2021, in °C
    icon = cat.ICON.ngc4008(zoom=zoom, time="P1D").to_dask()["tas"]
    return icon.sel(time="2021-07-15").squeeze().compute() - 273.15


fig, axes = plt.subplots(
    2, 2, figsize=(12, 7), subplot_kw={"projection": ccrs.PlateCarree()}
)
for ax, zoom in zip(axes.flat, [6, 7, 8, 9]):
    ax.set_extent([4, 16, 43, 49], crs=ccrs.PlateCarree())  # the Alps
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    image = egh.healpix_show(icon_day(zoom), ax=ax, cmap="RdYlBu_r", vmin=0, vmax=35)
    ax.set_title(f"zoom {zoom}")
fig.colorbar(image, ax=axes, shrink=0.7, label="temperature / °C")
fig.suptitle("ICON ngc4008, 2 m temperature, 15 July 2021")

fig.savefig("/app/pages/06_zoom_levels/maps_alps.png", dpi=100, bbox_inches="tight")
plt.show()

At zoom 6 the Alps are a few blue blocks; at zoom 9 you see the valleys and the ridges, and the cold of the
mountains next to the warm plains. A model that runs at kilometre scale gives you this detail, if you ask for
a fine enough zoom.

## 4. The global mean does not change

The global mean of ERA5 at every zoom from 0 (12 cells) to 8 (786 432 cells):

In [ ]:
global_means = {}
for zoom in range(9):
    global_means[zoom] = era5_mean(zoom)
    print(f"zoom {zoom}: {float(global_means[zoom].mean()):.3f} °C")

The same, up to 0.001 °C. Why? A cell of zoom 6 **is the mean of its 4 cells** of zoom 7, and so on: the
coarse zooms are made from the fine ones. The mean of means of equal areas is the mean of everything.

You can check it yourself. In the **nested** order of the cells, the 4 cells of zoom 7 inside cell $i$ of
zoom 6 are the cells $4i$, $4i+1$, $4i+2$ and $4i+3$: one after the other. So group the cells of zoom 7 by 4,
and average each group:

In [ ]:
zoom7 = global_means[7].values
made_zoom6 = zoom7.reshape(-1, 4).mean(axis=1)  # rows of 4 cells, mean of each row

print("cells:", zoom7.size, "->", made_zoom6.size)
print(
    f"largest difference to the zoom 6 of the catalog: {np.abs(made_zoom6 - global_means[6].values).max():.4f} °C"
)

## 5. One place does change

A cell is the mean over its whole area. So the value of the cell that contains a place depends on the zoom.
`hp.ang2pix` gives the cell of a place, as in class 05; `nside` is $2^z$:

In [ ]:
import healpix as hp

places = {"Hamburg": (10.0, 53.55), "Mont Blanc": (6.86, 45.83)}  # longitude, latitude
zooms = list(range(9))

fig, ax = plt.subplots(figsize=(8, 4))
for name, (lon, lat) in places.items():
    values = [
        float(
            global_means[z].values[hp.ang2pix(2**z, lon, lat, nest=True, lonlat=True)]
        )
        for z in zooms
    ]
    ax.plot(zooms, values, marker="o", label=name)
    print(name, [round(v, 1) for v in values])
ax.set_xlabel("zoom")
ax.set_ylabel("temperature / °C")
ax.set_title("ERA5 2 m temperature of the cell containing the place, 2021–2023")
ax.legend()

fig.savefig("/app/pages/06_zoom_levels/point_zooms.png", dpi=100, bbox_inches="tight")
plt.show()

Hamburg is flat country: from zoom 3 on, its cell has almost the same temperature. The Mont Blanc is not:
its cell mixes the summit (about 4 800 m) with valleys, and the value jumps from zoom to zoom. Even at zoom 8 the cell
is 25 km wide and much warmer than the summit itself: a station measures a point, a cell an area (class 05).

## 6. Choosing a zoom

| You want | Zoom | One map |
|---|---|---|
| a global mean, a time series of a large region | any, e.g. 0–4: the mean does not change | tiny |
| a map of the globe | 5–6 | 0.2 MB |
| a map of a country or a mountain range | 8–9 | 3–13 MB |
| one place in the mountains or at the coast | as fine as the data has | |

Take the **coarsest zoom that shows what you need**: the data, and the time to get it, grow by 4 with every
zoom level.

## 7. Exercise: your region

Take the region of your page of class 01 (or another one you like):

1. Maps of your region at four zooms, as in section 3 (choose the zooms; change `set_extent`). Saved as
   `/app/pages/06_zoom_levels/region_exercise.png`.
2. The value of the cell of one place in your region at zoom 0 to 8, as in section 5. Saved as
   `/app/pages/06_zoom_levels/point_exercise.png`.
3. Which zoom do you need to see what interests you in your region?

In [ ]:
# 1. maps of your region at four zooms

In [ ]:
# 2. one place at zoom 0 to 8

## 8. Your page

Open `pages/06_zoom_levels/my_page.rst` in JupyterLab, write your name, and describe what the zoom changes.
Look at it on http://localhost:8000 → *Classes → 06*.